# Milestone 2 — Unified Music Similarity Evaluation

Compare **original 518-dimensional FMA features**, **16-dimensional PCA**, and **16-dimensional feature-based autoencoder embeddings** under the **same retrieval protocol**: test queries against training candidates, cosine nearest neighbors, `k=5` and `k=10`. Evaluate genre consistency, cross-genre rate, tag overlap and tag coverage.

**Prerequisites:** commit `scripts/pca_baseline.py`, `scripts/retrieval.py`, and `scripts/evaluate.py` to your repository. The processed `training.npz`, `validation.npz`, `test.npz`, and `metadata.csv` must exist. The autoencoder embeddings must be exported separately from the **same trained autoencoder** using the instructions below. Do not interpret missing tags as zero similarity.

## 1. Set up repository and imports

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_NAME = "music-similarity-representation-learning"
REPO_URL = "https://github.com/samiyaalizaidi/music-similarity-representation-learning.git"

if "google.colab" in sys.modules:
    PROJECT_ROOT = Path("/content") / REPO_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "data" / "processed").is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Run this notebook from inside your repository.")

PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.pca_baseline import fit_pca, transform_pca
from scripts.retrieval import retrieve_neighbors
from scripts.evaluate import evaluate_retrieval

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results" / "milestone2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
K_VALUES = [5, 10]
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)

## 2. Load aligned processed features and metadata

In [ ]:
def load_split(name):
    with np.load(DATA_DIR / f"{name}.npz", allow_pickle=False) as data:
        X = data["X"].astype(np.float32)
        ids = data["track_ids"].astype(np.int64)
    assert X.ndim == 2 and len(X) == len(ids)
    return X, ids

X_train, train_ids = load_split("training")
X_val, val_ids = load_split("validation")
X_test, test_ids = load_split("test")
metadata = pd.read_csv(DATA_DIR / "metadata.csv", index_col=0)
metadata.index = pd.to_numeric(metadata.index, errors="raise").astype(np.int64)
assert metadata.index.is_unique
assert set(train_ids).issubset(metadata.index)
assert set(test_ids).issubset(metadata.index)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Metadata columns:", list(metadata.columns))

GENRE_COLUMN = "genre" if "genre" in metadata.columns else "genre_top"
if GENRE_COLUMN not in metadata.columns:
    raise KeyError("Expected genre or genre_top in metadata.csv")
TAGS_COLUMN = "tags" if "tags" in metadata.columns else ("track_tags" if "track_tags" in metadata.columns else "tags")
if TAGS_COLUMN not in metadata.columns:
    print("WARNING: No tags column. Tag overlap metrics will be missing (NaN).")

## 3. Original-feature and PCA representations

In [ ]:
embeddings = {
    "original": {"train": X_train, "test": X_test}
}
pca = fit_pca(X_train, n_components=16, seed=SEED)
embeddings["pca"] = {
    "train": transform_pca(pca, X_train),
    "test": transform_pca(pca, X_test),
}
print("PCA retained variance:", round(float(pca.explained_variance_ratio_.sum()), 4))

## 4. Autoencoder embeddings

The model in your previous `music_similarity_updated(2).ipynb` exposes `ae.transform(X)` **after** `ae.fit(...)`. To reuse that trained model, run the following code **in that original notebook, immediately after the training cell** (where `ae`, `X_train`, `X_test`, `train_ids`, `test_ids`, and `PROCESSED_DIR` are already defined):

```python
np.savez_compressed(
    PROCESSED_DIR / "ae_embeddings.npz",
    train=ae.transform(X_train).astype(np.float32),
    test=ae.transform(X_test).astype(np.float32),
    train_ids=train_ids,
    test_ids=test_ids,
)
```

**Colab note:** If the export is created in a different Colab session, download/commit it or otherwise copy it into this notebook's `data/processed/` folder. The same saved file must contain both train and test embeddings. Never independently train two autoencoders for the two splits.

In [ ]:
ae_path = DATA_DIR / "ae_embeddings.npz"
if not ae_path.exists():
    raise FileNotFoundError(
        f"Missing {ae_path}. Export train and test embeddings from the trained "
        "autoencoder notebook using the instructions above."
    )
with np.load(ae_path, allow_pickle=False) as data:
    ae_train = data["train"].astype(np.float32)
    ae_test = data["test"].astype(np.float32)
    ae_train_ids = data["train_ids"].astype(np.int64)
    ae_test_ids = data["test_ids"].astype(np.int64)
if not np.array_equal(ae_train_ids, train_ids) or not np.array_equal(ae_test_ids, test_ids):
    raise ValueError("AE embedding track IDs/order do not match processed splits")
if len(ae_train) != len(train_ids) or len(ae_test) != len(test_ids):
    raise ValueError("AE embedding row counts do not match split lengths")
embeddings["autoencoder"] = {"train": ae_train, "test": ae_test}
print("AE embeddings:", ae_train.shape, ae_test.shape)

## 5. Same retrieval and metrics for every representation

In [ ]:
all_metrics, all_retrievals, all_detailed = [], [], []
for name, data in embeddings.items():
    neighbors = retrieve_neighbors(
        query_embeddings=data["test"],
        candidate_embeddings=data["train"],
        query_ids=test_ids,
        candidate_ids=train_ids,
        k=max(K_VALUES),
    )
    neighbors["representation"] = name
    all_retrievals.append(neighbors)
    for k in K_VALUES:
        metrics, detailed = evaluate_retrieval(
            neighbors, metadata, k=k,
            genre_column=GENRE_COLUMN, tags_column=TAGS_COLUMN,
        )
        metrics["representation"] = name
        all_metrics.append(metrics)
        detailed["representation"] = name
        detailed["k"] = k
        all_detailed.append(detailed)

comparison = pd.DataFrame(all_metrics)
retrieval_results = pd.concat(all_retrievals, ignore_index=True)
detailed_results = pd.concat(all_detailed, ignore_index=True)
display(comparison.round(4))
comparison.to_csv(RESULTS_DIR / "comparison.csv", index=False)
retrieval_results.to_csv(RESULTS_DIR / "retrieval_results.csv", index=False)
detailed_results.to_csv(RESULTS_DIR / "detailed_results.csv", index=False)
print("Saved tables to", RESULTS_DIR)

## 6. Plots

In [ ]:
results_k10 = comparison.query("k == 10").set_index("representation")
ax = results_k10[["genre_consistency", "cross_genre_rate"]].plot.bar(figsize=(9, 5), rot=0)
ax.set(title="Genre consistency vs cross-genre retrieval (k=10)", ylabel="Fraction", ylim=(0, 1))
plt.tight_layout()
plt.savefig(RESULTS_DIR / "genre_comparison.png", dpi=200)
plt.show()

if results_k10["tag_coverage"].gt(0).any():
    ax = results_k10[["tag_overlap", "cross_genre_tag_overlap"]].plot.bar(figsize=(9, 5), rot=0)
    ax.set(title="Tag overlap on evaluable pairs (k=10)", ylabel="Jaccard similarity")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "tag_comparison.png", dpi=200)
    plt.show()
else:
    print("No evaluable tag pairs; inspect metadata and tag coverage.")

## 7. Cross-genre examples and metadata coverage

In [ ]:
examples = detailed_results.query("representation == 'autoencoder' and k == 10").copy()
examples = examples[examples["cross_genre"] == 1.0].copy()
for column in ["title", "artist"]:
    if column in metadata.columns:
        examples[f"query_{column}"] = examples["query_id"].map(metadata[column])
        examples[f"neighbor_{column}"] = examples["neighbor_id"].map(metadata[column])
cols = [c for c in ["query_id", "query_title", "query_artist", "query_genre", "neighbor_id", "neighbor_title", "neighbor_artist", "neighbor_genre", "cosine_similarity", "tag_overlap"] if c in examples.columns]
display(examples[cols].head(20))
examples.to_csv(RESULTS_DIR / "cross_genre_examples.csv", index=False)
display(comparison[["representation", "k", "tag_coverage", "cross_genre_tag_coverage"]].round(4))

## Interpretation reminders

- A higher **cross-genre rate** is not automatically better musical similarity.
- Compare **tag overlap** only where metadata is available; include tag coverage.
- All three methods now use **the same test queries and training candidate tracks**.
- The precomputed-feature models do **not** directly learn from MP3 waveforms; this milestone follows the originally proposed feature-based comparison.
- For qualitative evaluation, listen to selected cross-genre neighbor pairs and record shared rhythm, timbre, instrumentation, or energy.